# Query DSL — operator tour

This notebook walks through every operator in the query DSL
against the live corpus. Each section runs a small batch of
queries and tabulates the matching clip counts so you can see
what each operator does.

> New to the DevKit? Start with `01_quickstart.ipynb` for an
> introduction to the dataset and the DSL's place in it.

### Grammar in one screen

| form | meaning |
|---|---|
| `<entity>.<attr> <cmp> <value>` | basic attribute predicate |
| `<entity>(<expr>, …)` | same-entity grouping |
| `and`, `or`, `not` | boolean composition |
| `A while B` | A and B intersect, including endpoint-only touch |
| `A while_strict B` | A and B overlap for positive duration |
| `A then(K) B` | B starts during A or at most K seconds after A |
| `A before(K) B` | A ends strictly before B; optional maximum gap K |
| `A because_of B` | A's `because_of` edge points at a B |
| `A influenced_by B` | A's `Influence` side-channel points at a B |
| `within W: E` | restrict E's time window to W's intervals |

The full specification lives in `docs/user/query_language.md`.


## Setup


In [1]:
import os
from pathlib import Path

from cascade_av.dataset import CascadeDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [2]:
ds = CascadeDataset(Path(os.environ["CASCADE_AV_DATASET_ROOT"]))

def run(queries: list[str]) -> pd.DataFrame:
    """Run a batch of DSL queries and tabulate clip counts."""
    return pd.DataFrame(
        [(q, ds.count(q)) for q in queries],
        columns=["query", "matching clips"],
    )


## Attribute predicates

The atomic predicate is `entity.attribute <cmp> value`. The
comparison is `=` for equality and `<`, `<=`, `>`, `>=` for
ordered attributes.


In [3]:
run([
    "agent.type = ped",
    "ego.action = decel",
    "light.color = red",
    "env.lanes >= 2",
])


,query,matching clips
0,agent.type = ped,46
1,ego.action = decel,14
2,light.color = red,33
3,env.lanes >= 2,56


## Hierarchical aliases

Several attribute values are *parent aliases* that expand to a
union of children: `vehicle` (car, truck, bus, …),
`vru` (pedestrian, cyclist), `intersection` (4-way, T-junction,
roundabout, …), `turn` (left, right), `change_lane`
(change-left, change-right).


In [4]:
run([
    "agent.type = vehicle",
    "agent.type = vru",
    "agent.type = cyclist",
    "env.type = intersection",
])


,query,matching clips
0,agent.type = vehicle,89
1,agent.type = vru,48
2,agent.type = cyclist,14
3,env.type = intersection,71


## Set membership

`attr in (a, b, c)` is sugar for an OR over the listed values.


In [5]:
run([
    "ego.action in (stop, yield, decel)",
    "agent.type in (ped, cyclist, animal)",
])


,query,matching clips
0,"ego.action in (stop, yield, decel)",68
1,"agent.type in (ped, cyclist, animal)",48


## Same-entity coupling

Inside an `agent(…)` clause, every constraint applies to the
**same** agent. Free-floating predicates may bind to different
agents — note how the count changes.


In [6]:
run([
    "agent(type = vehicle, pos = front)",
    "agent.type = vehicle and agent.pos = front",
])


,query,matching clips
0,"agent(type = vehicle, pos = front)",86
1,agent.type = vehicle and agent.pos = front,87


## Ego-relative direction — `agent.dir`

Direction aliases: `same`, `opposite`, `perpendicular_lr`,
`perpendicular_rl`, and the parent `perpendicular` (union of
the two perpendicular leaves). Like `agent.pos`, `agent.dir`
samples at the agent's visibility-window midpoint.


In [7]:
run([
    "agent.dir = same",
    "agent.dir = opposite",
    "agent.dir = perpendicular",
    # Oncoming vehicle — the natural way to ask "is there an
    # oncoming car?" for unprotected-turn analysis.
    "agent(type = vehicle, dir = opposite)",
])


,query,matching clips
0,agent.dir = same,85
1,agent.dir = opposite,46
2,agent.dir = perpendicular,52
3,"agent(type = vehicle, dir = opposite)",33


## Per-interval pose — `pos_any` / `dir_any`

`agent.pos` / `agent.dir` sample at the **midpoint** of the
agent's visibility window (one deterministic value per
agent). For agents whose pose changes mid-window — e.g. a
vehicle that approaches in front and ends up perpendicular as
it crosses an intersection — the midpoint misses the
transient. `pos_any` / `dir_any` are `kind="list"` and match
if **any** `EgoRelativePose` interval carried the value.

The delta between the two flavors shows how often pose
changes during visibility:


In [8]:
# Midpoint vs interval — same query, different cardinality.
run([
    "agent.dir = perpendicular",      # midpoint
    "agent.dir_any = perpendicular",  # any interval
])


,query,matching clips
0,agent.dir = perpendicular,52
1,agent.dir_any = perpendicular,55


In [9]:
# Composite: vehicles that were either in front OR crossing
# at some point during their visibility.
run([
    "agent(type = vehicle, "
    "pos_any in (front, perpendicular_lr, perpendicular_rl))",
])


,query,matching clips
0,"agent(type = vehicle, pos_any in (front, perpendicular_lr, perpendicular_rl))",86


## Boolean operators

Standard `and`, `or`, `not` over any sub-expression.


In [10]:
run([
    "agent.type = ped and ego.action in (stop, yield, decel)",
    "ego.action = stop or ego.action = yield",
    "agent.type = ped and not ego.action = drive",
])


,query,matching clips
0,"agent.type = ped and ego.action in (stop, yield, decel)",36
1,ego.action = stop or ego.action = yield,61
2,agent.type = ped and not ego.action = drive,7


## Action flags

Action types in the schema 2.0.0 corpus encode flags as
parenthesized suffixes on the `action_type` string itself
(`oxd:Walk (jaywalk)`). The legacy flag-field syntax
`action(jaywalk = true)` no longer parses — match the
suffix variants directly via literal-string equality or
`in (...)`.

Post-0.6.1, `Jaywalk` is also a standalone action (alias
`jaywalk_action`); the combined-suffix forms are deprecated but
still match, so a complete jaywalk query enumerates both.


In [11]:
run([
    "agent(type = ped, action.type in ("
    '"Jaywalk", '
    '"oxd:Walk (jaywalk)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (jaywalk)", "oxd:Run (jaywalk, erratic)"))',
    "agent.action.type in ("
    '"oxd:Walk (erratic)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (erratic)", "oxd:Run (jaywalk, erratic)")',
])


,query,matching clips
0,"agent(type = ped, action.type in (""Jaywalk"", ""oxd:Walk (jaywalk)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run...",17
1,"agent.action.type in (""oxd:Walk (erratic)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run (erratic)"", ""oxd:Run (...",0


## Temporal — `while`

`A while B` keeps only pairs of matches whose intervals
intersect. Intervals are closed, so endpoint-only touch counts.


In [12]:
run([
    "agent.type = ped while ego.action = decel",
    "light.color = red while ego.action = stop",
])


,query,matching clips
0,agent.type = ped while ego.action = decel,4
1,light.color = red while ego.action = stop,17


## Temporal — `while_strict`

`A while_strict B` requires a positive-duration intersection.
Unlike `while`, two intervals that only touch at one endpoint
do not match.


In [13]:
run([
    "agent.type = ped while_strict ego.action = decel",
    "light.color = red while_strict ego.action = stop",
])


,query,matching clips
0,agent.type = ped while_strict ego.action = decel,4
1,light.color = red while_strict ego.action = stop,17


## Temporal — `then(K)`

`A then(K) B` is the permissive sequence operator: B may start
during A, touch A's endpoint, or start within K seconds after A.
Bare `then` (no parens) defaults to K = 0 — overlap or touch only.


In [14]:
run([
    "light.color = yellow then(3) ego.action = stop",
    "light.color = green then ego.action = drive",
])


,query,matching clips
0,light.color = yellow then(3) ego.action = stop,6
1,light.color = green then ego.action = drive,16


## Temporal — `before(K)`

`A before(K) B` is strict: A must end before B starts, with a
positive gap no larger than K. Overlap and endpoint touch do not
match. Bare `before` omits the upper bound and accepts any
positive gap.


In [15]:
run([
    "light.color = yellow before(3) ego.action = stop",
    "light.color = yellow before ego.action = stop",
])


,query,matching clips
0,light.color = yellow before(3) ego.action = stop,3
1,light.color = yellow before ego.action = stop,5


## Relational — `because_of`

Follows the schema's `because_of` causal edge. Both halves are
emitted as match tuples so you can inspect cause and effect.


In [16]:
run([
    # `because_of` holds the IDs of *actions / states / objects*
    # that caused an action — never agent IDs directly. Point at
    # the agent's action, a signal state, or a traffic object.
    "ego.action = stop because_of light.color = red",
    "ego.action in (stop, yield, decel) because_of "
    'agent.action.type in ("oxd:Walk", "oxd:Stand")',
])


,query,matching clips
0,ego.action = stop because_of light.color = red,10
1,"ego.action in (stop, yield, decel) because_of agent.action.type in (""oxd:Walk"", ""oxd:Stand"")",17


## Relational — `influenced_by`

Walks the schema's `Influence.influencers` side-channel (distinct
from action-rooted `because_of`): "this entity *was under the
influence of* B". The LHS is a bare entity (`ego` / `agent(...)`)
and the match interval is the influence window, so it composes
with `while`, `while_strict`, `then`, and `before`. A
`light.color` RHS is scoped to that window.


In [17]:
run([
    "ego influenced_by light.color = red",
    "agent(type = vehicle) influenced_by light.color = red",
    "ego influenced_by (obj.type = stop_sign or light.color = red)",
])


,query,matching clips
0,ego influenced_by light.color = red,30
1,agent(type = vehicle) influenced_by light.color = red,18
2,ego influenced_by (obj.type = stop_sign or light.color = red),37


## Scoping — `within W: E`

Restricts `E`'s evaluation window to `W`'s intervals. Inside,
`not E` means *E does not happen during W* — useful for
negative scenarios like "during red, ego never stops."


In [18]:
run([
    "within light.color = red: not ego.action = stop",
    "within env.type = road: agent.type = ped",
])


,query,matching clips
0,within light.color = red: not ego.action = stop,16
1,within env.type = road: agent.type = ped,46


## Annotator-tagged flags

Some attributes are not derived from kinematics but tagged by
the annotator (e.g. *"the ego could have safely cleared this
yellow"*). They compose with the same DSL.


In [19]:
run([
    "light(color = yellow, ego_in_on_yellow = true)",
    "light(color = yellow, could_have_cleared = true)",
])


,query,matching clips
0,"light(color = yellow, ego_in_on_yellow = true)",12
1,"light(color = yellow, could_have_cleared = true)",0


## Agent group size — `agent.amount`

Lone agent vs row/group vs traffic-density tier. Aliases:
`single`, `row` (alias `group`), `light_traffic`,
`medium_traffic`, `heavy_traffic`, plus the parents
`multiple` (everything but `single`) and `traffic` (the
three density tiers).


In [20]:
run([
    "agent.amount = single",
    "agent.amount = group",
    "agent(type = ped, amount = group)",
])


,query,matching clips
0,agent.amount = single,89
1,agent.amount = group,69
2,"agent(type = ped, amount = group)",30


## Signaling details on `agent.prop` / `ego.prop`

Properties of type `signal` carry a `signaling_details`
sub-object exposing the source modality (flashing light,
hand gesture, holding sign, other) and a `not_facing_ego`
boolean. Non-signal properties resolve to `None` so the
predicates fall through cleanly.


In [21]:
run([
    "agent.prop.source = flashing_light",
    "agent.prop.source in (flashing_light, holding_sign, other)",
    "agent.prop.not_facing_ego = false",
    "agent(prop.source = flashing_light)",
])


,query,matching clips
0,agent.prop.source = flashing_light,50
1,"agent.prop.source in (flashing_light, holding_sign, other)",53
2,agent.prop.not_facing_ego = false,52
3,agent(prop.source = flashing_light),50


## Containment flags — `lane_edge`, `near_lane`, `illegal_lane`

Boolean predicates that match existentially across the
entity's containment records. Exposed on both `agent` and
`obj`. `lane_edge` is true when any containment carries
`edge ∈ {"left","right"}`; `near_lane` and `illegal_lane`
mirror the boolean flags directly.


In [22]:
run([
    "agent.illegal_lane = true",
    "agent.lane_edge = true",
    "obj.lane_edge = true",
    "agent(type = car, illegal_lane = true)",
])


,query,matching clips
0,agent.illegal_lane = true,8
1,agent.lane_edge = true,46
2,obj.lane_edge = true,89
3,"agent(type = car, illegal_lane = true)",1


## Clip-level attributes — `clip.eventful` / `clip.eventful_reason`

The boolean `clip.eventful` flags clips worth a closer look.
Post-0.6.1, `clip.eventful_reason` is queryable alongside it —
values: `ego_adapts`, `special_env`, `agent_adapts`, `other`.


In [23]:
run([
    "clip.eventful = true",
    "clip.eventful_reason = ego_adapts",
])


,query,matching clips
0,clip.eventful = true,99
1,clip.eventful_reason = ego_adapts,0


## Newly-aliased `"Other"` / `"Vehicle"` values

Four families that previously required the schema-literal
escape hatch (`agent.type = "Other"`) now resolve `other`
as a friendly alias. The annotator's generic-vehicle
fallback `"Vehicle"` (used when an agent is a vehicle that
doesn't match any specific subtype) is folded into the
`vehicle` parent alias and also reachable as the leaf
`generic_vehicle`.


In [24]:
run([
    "agent.type = other",
    "agent.type = generic_vehicle",
    "env.type = other",
    "agent.action.type = other",
])


,query,matching clips
0,agent.type = other,7
1,agent.type = generic_vehicle,4
2,env.type = other,21
3,agent.action.type = other,6


## Condition → environment lookup — `cond.env_*`

Each `Condition` carries an `env_id` pointing at one of the
bundle's `Environment` records. The DSL has no cross-entity
join operator today, so the four most-useful environment
fields are exposed as *denormalized* attributes on `cond`:
`cond.env_type` (reuses the `env_type` alias family),
`cond.env_lanes`, `cond.env_one_way`, and `cond.env_id`
(raw passthrough). A condition with an unresolvable
`env_id` falls through to `None` for the derived fields, so
predicates compose without an explicit guard.


In [25]:
run([
    "cond.env_type = road",
    "cond.type = construction and cond.env_type = road",
    "cond.env_one_way = true",
    "cond.env_lanes > 2",
])


,query,matching clips
0,cond.env_type = road,55
1,cond.type = construction and cond.env_type = road,20
2,cond.env_one_way = true,15
3,cond.env_lanes > 2,10
